# Pipeline đơn giản: Random Forest + CNN 1D

Notebook này dùng cho JupyterLab để chạy lại và minh hoạ kết quả đồ án. Mục tiêu là giữ pipeline dễ giải thích: RF trên feature thủ công và CNN 1D trên chuỗi cảm biến.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RESULT_PATH = PROJECT_ROOT / 'reports' / 'simple_results.json'
FIG_DIR = PROJECT_ROOT / 'reports' / 'figures'
CONFIG_PATH = PROJECT_ROOT / 'experiments' / 'configs' / 'SIMPLE_CNN.yaml'

PROJECT_ROOT

## 1. Chạy lại pipeline

Mặc định cell dưới đây không train lại CNN để tránh mất thời gian. Nếu muốn chạy lại toàn bộ, đổi `RUN_TRAIN = True`. Nếu chỉ muốn cập nhật RF và biểu đồ, đổi `RUN_RF_ONLY = True`.

In [ ]:
RUN_TRAIN = False
RUN_RF_ONLY = False

if RUN_TRAIN:
    subprocess.run([
        sys.executable,
        '-m', 'scripts.run_simple_pipeline',
        '--config', str(CONFIG_PATH),
    ], cwd=PROJECT_ROOT, check=True)
elif RUN_RF_ONLY:
    subprocess.run([
        sys.executable,
        '-m', 'scripts.run_simple_pipeline',
        '--config', str(CONFIG_PATH),
        '--skip-cnn',
    ], cwd=PROJECT_ROOT, check=True)

subprocess.run([sys.executable, '-m', 'scripts.plot_simple_results'], cwd=PROJECT_ROOT, check=True)

## 2. Bảng kết quả chính

In [ ]:
with open(RESULT_PATH) as f:
    results = json.load(f)

summary = pd.DataFrame([
    {
        'Mô hình': 'CNN 1D simple',
        'Macro-F1': results['cnn']['mean_macroF1'],
        'Std': results['cnn']['std_macroF1'],
        'Accuracy': None,
    },
    {
        'Mô hình': 'Random Forest',
        'Macro-F1': results['rf']['mean_macroF1'],
        'Std': results['rf']['std_macroF1'],
        'Accuracy': results['rf']['mean_accuracy'],
    },
])

summary.style.format({'Macro-F1': '{:.4f}', 'Std': '{:.4f}', 'Accuracy': '{:.4f}'})

## 3. Biểu đồ để đưa vào báo cáo

In [ ]:
for filename in [
    'simple_model_comparison.png',
    'simple_fold_macro_f1.png',
    'simple_rf_accuracy.png',
]:
    path = FIG_DIR / filename
    print(path.relative_to(PROJECT_ROOT))
    display(Image(filename=str(path)))

## 4. Kết luận ngắn để thuyết trình

- Random Forest đạt Macro-F1 cao hơn CNN 1D trong cấu hình đơn giản.
- RF dễ giải thích vì dựa trên feature thống kê, FFT, tương quan và SMA.
- CNN 1D vẫn nên giữ như mô hình học sâu bổ sung vì học trực tiếp từ chuỗi cảm biến.
- Không cần trình bày ResNet, TCN, BiLSTM, OneCycle, MixUp/CutMix ở phần chính vì làm đồ án khó theo dõi hơn.